# Deep Learning Model Selection and Training

`(update)`
In this notebook, we will focus exclusively on Phase 1 testing and selection for Deep Learning:
- Validate heavy Deep Learning Baselines (Inception V3, VGG16, DenseNet201).
- Perform a grid search across Transfer Learning Initialization Strategies for MobileViT.
- Execute runs using the unified Lightning `train_dl.py` scripts and `run_grid_search.py`.

## Environment Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# ruff: noqa: F821
from jute_disease.utils.constants import DEFAULT_SEED
from jute_disease.utils.logger import get_logger
from jute_disease.utils.seed import seed_everything

In [ ]:
logger = get_logger(__name__)
seed_everything(DEFAULT_SEED)

In [ ]:
!git clone https://github.com/qu1r0ra/jute-disease-detection.git
%cd jute-disease-detection

%pip install uv

# Install dependencies into the Colab runtime, not into a virtual environment (.venv)
logger.info("Installing dependencies with uv...")
!uv pip install --system -e .

Mount your Google Drive into the Colab runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

If you haven't yet,

1. Download `data.zip` from <https://drive.google.com/drive/folders/1WoQ-Xzy0Prl9lInHW5JpGX4tpE9YDUua?usp=sharing> and upload it to your Google Colab account's Google Drive. You can simply upload it to the root of _My Drive_ for simplicity.
2. Update `DATA_ZIP_PATH` below to the path where you stored the file. If you uploaded it to the root of _My Drive_, you can set it to **"/content/drive/MyDrive/data.zip"**.

In [ ]:
# Update this to where you stored data.zip in your GDrive.
# For organization, we stored ours in
# "/content/drive/MyDrive/Colab/Jute Leaf Disease/data.zip"
DATA_ZIP_PATH = "/content/drive/MyDrive/Colab/Jute Leaf Disease/data.zip"

if Path(DATA_ZIP_PATH).exists():
    logger.info(f"Unzipping {DATA_ZIP_PATH}...")
    !unzip -q -n "$DATA_ZIP_PATH" -d .
    logger.info("Data unpacked.")
else:
    logger.warning(
        f"Zip file not found at {DATA_ZIP_PATH}. "
        "Please check the path or upload your data."
    )

To persist our training artifacts beyond the Colab VM, we can _symlink_ the `artifacts` folder directly to our Google Drive.

In [ ]:
GDRIVE_PATH = Path(DATA_ZIP_PATH).parent
GDRIVE_ARTIFACTS = GDRIVE_PATH / "artifacts"
LOCAL_ARTIFACTS = Path("artifacts")

GDRIVE_ARTIFACTS.mkdir(parents=True, exist_ok=True)

if not LOCAL_ARTIFACTS.exists() and not LOCAL_ARTIFACTS.is_symlink():
    LOCAL_ARTIFACTS.symlink_to(GDRIVE_ARTIFACTS)
    logger.info(f"Symlinked {LOCAL_ARTIFACTS.absolute()} -> {GDRIVE_ARTIFACTS}")
else:
    logger.info(f"{LOCAL_ARTIFACTS} already exists or is linked.")

Let us perform a quick sanity test to ensure all generated files show up inside your Google Drive folder containing your `data.zip`. If you see a generated `test.txt` file then you are all set to proceed.

In [ ]:
test_file = LOCAL_ARTIFACTS / "test.txt"
test_file.write_text("Hacking the mainframe.")

if (GDRIVE_ARTIFACTS / "test.txt").exists():
    logger.info("Symlink worked.")
    test_file.unlink()
else:
    logger.error("Symlink failed :<")

## Transfer Learning Setup

We saw from EDA that our dataset is pretty small (2382 images across 6 classes) for an image recognition task. To address our limitation in data, we decided to employ transfer learning as a key technique (among others, such as data augmentation) for our deep learning experiments.

(levels of transfer learning)
1. **Level 1**: ImageNet -> our Jute dataset
2. **Level 2**: ImageNet -> PlantVillage -> our Jute dataset
3. **Level 3**: ImageNet -> PlantVillage -> PlantDoc -> our Jute dataset

Levels 2 and 3 are known as **multistage transfer learning (MSTL)**, which as the name suggests, is transfer learning with multiple stages. An analogy of TL vs. MSTL can be made in the task of teaching a Tagalog-speaking Filipino the Spanish language. While we could just teach them Spanish directly (TL), it may be more effective to first teach them Chavacano (a Spanish-based language spoken in the Philippines) before teaching them Spanish (MSTL). Perhaps learning Chavacano first will make learning Spanish a lot easier, leading to a greater Spanish proficiency by the end.

Thus, we hope that transfer learning will enable our deep learning models to adapt general patterns learned from ImageNet objects to the domain of leaf disease detection. We are also curious as to whether utilizing MSTL with similar but general datasets such as PlantVillage and PlantDoc can improve performance.

Specifically, we will experiment with six (6) established deep learning architectures:
- InceptionV3
- VGG16
- DenseNet201
- ResNet
- MobileNet
- MobileViT

`(i will polish later)`

## Deep Learning Baselines (Level 1: ImageNet Only)

We systematically train and evaluate our six chosen architectures on the `DATA_ZIP_PATH` jute splits. Every model's feature extractor initiates from ImageNet generic representations. We will freeze their backbones and only train the final custom dense classifiers.

In [ ]:
import subprocess

baseline_models = [
    "inception.yaml",
    "vgg.yaml",
    "densenet.yaml",
    "resnet.yaml",
    "mobilenet.yaml",
    "mobilevit.yaml",
]

for model_config in baseline_models:
    logging.info(
        "\n{'=' * 50}\n"
        f"Training {model_config} baseline directly on Jute...\n"
        "{'=' * 50}"
    )
    cmd = [
        "uv",
        "run",
        "python",
        "scripts/train_dl.py",
        "fit",
        "--config",
        f"configs/baselines/{model_config}",
    ]
    subprocess.run(cmd, check=True)

## === Everything above is final ===

## 2. MSTL Domain Initializations (Pre-training)
We now download the massive `PlantVillage` and specialized `PlantDoc` datasets via KaggleHub to execute the Multi-Stage Transfer Learning on our MobileViT engine.

In [ ]:
from jute_disease.data.download import download_plant_doc, download_plant_village

# Download external datasets directly into Colab environment
download_plant_village()
download_plant_doc()

**Pre-Train MobileViT on PlantVillage (Produces Level 2 Checkpoint)**
We use our custom PyTorch Lightning pre-training script on PlantVillage. Early-stopping is implemented intrinsically (Defaults to 50 epochs, halts upon val_loss convergence).

In [ ]:
!uv run python src/jute_disease/engines/dl/pretrain.py \
  --data_dir data/external/plantvillage \
  --output_path artifacts/checkpoints/pretrained/mobilevit_plantvillage.ckpt

**Pre-Train MobileViT on PlantDoc (Produces Level 3 Checkpoint)**
Note the `--base_weights` parameter: We resume *exactly* from the Level 2 checkpoint! This synthesizes the entire ImageNet -> PlantVillage -> PlantDoc hierarchy!

In [ ]:
!uv run python src/jute_disease/engines/dl/pretrain.py \
  --data_dir data/external/plantdoc \
  --base_weights artifacts/checkpoints/pretrained/mobilevit_plantvillage.ckpt \
  --output_path artifacts/checkpoints/pretrained/mobilevit_plantvillage_plantdoc.ckpt

## 3. Transfer Learning Grid Search (MobileViT Evaluation)
Running our Grid Search config utilizing the newly synthesized checkpoints for MobileViT, quantifying the precise benefits of each Initialization Level on small-dataset Leaf Disease recognition!

In [ ]:
!uv run python scripts/run_grid_search.py configs/grid/mobilevit_grid.yaml

## 4. WandB Analysis
From Weights & Biases, we can now deduce our Champion architectural baseline, as well as definitively prove whether or not Level 3 MSTL was superior computationally versus Level 1 generic pretraining.